In [29]:
from pathlib import Path

from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import json
import seaborn as sns
def normalize(text):
    return " ".join(text.lower().replace("ё", "е").split())


def load_embeddings(folder):
    arrays = []
    sources = []

    for path in sorted(Path(folder).glob("*.npy")):
        vectors = np.load(path, allow_pickle=False)

        chunks = json.loads(
            path.with_suffix(".chunks.json").read_text(encoding="utf-8")
        )

        if vectors.ndim != 2 or len(vectors) != len(chunks):
            raise ValueError(f"Некорректный файл: {path}")

        arrays.append(vectors)

        for chunk_id, text in enumerate(chunks):
            sources.append({
                "video_id": path.stem,
                "chunk_id": chunk_id,
                "text": text,
            })

    if not arrays:
        raise ValueError(f"Нет эмбеддингов: {folder}")

    return np.concatenate(arrays), sources


def check_model(folder, tests, query_vectors):
    vectors, sources = load_embeddings(folder)
    details = []

    for test, query_vector in zip(tests, query_vectors):
        # Правильными считаем фрагменты нужного видео,
        # содержащие хотя бы одну эталонную цитату целиком.
        relevant = {
            index
            for index, source in enumerate(sources)
            if source["video_id"] == test["video_id"]
            and any(
                normalize(quote) in normalize(source["text"])
                for quote in test["evidence"]
            )
        }

        if not relevant:
            details.append({
                "question": test["question"],
                "hit1": 0,
                "hit3": 0,
                "status": "evidence_not_contained",
                "found": [],
            })
            continue

        scores = vectors @ query_vector
        top = np.argsort(scores)[::-1][:3]

        details.append({
            "question": test["question"],
            "hit1": int(int(top[0]) in relevant),
            "hit3": int(any(int(index) in relevant for index in top)),
            "found": [
                {
                    **sources[int(index)],
                    "score": float(scores[index]),
                }
                for index in top
            ],
        })

    return {
        "hit1": sum(item["hit1"] for item in details) / len(details),
        "hit3": sum(item["hit3"] for item in details) / len(details),
        "missing_evidence": sum(
        item.get("status") == "evidence_not_contained"
        for item in details
    ),
        "details": details,
    }
def download_model(model_path):
    return SentenceTransformer(
        str(model_path),
        device="mps",
        local_files_only=True,
    )


def split_text(model, text, chunk_size=512, overlap=64):
    tokenizer = model.tokenizer

    limit = (
        model.max_seq_length
        - tokenizer.num_special_tokens_to_add(pair=False)
    )
    chunk_size = min(chunk_size, limit)

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть больше нуля")

    overlap = min(overlap, chunk_size // 4)

    if overlap < 0:
        raise ValueError("overlap не может быть отрицательным")

    step = chunk_size - overlap
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    for start in range(0, len(token_ids), step):
        end = start + chunk_size

        chunks.append(
            tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True,
            )
        )

        if end >= len(token_ids):
            break

    return chunks


def embedings(model, text_file, transcrip_path, save_path,chunk_size,overlap):
    file_path = Path(transcrip_path) / f"{text_file}.txt"
    text = file_path.read_text(encoding="utf-8")

    chunks = split_text(
    model=model,
    text=text,
    chunk_size=chunk_size,
    overlap=overlap,
)
    if not chunks:
        print(f"Пустой текст: {file_path}")
        return

    vectors = model.encode(
        chunks,
        batch_size=2,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    save_path = Path(save_path)
    save_path.mkdir(parents=True, exist_ok=True)

    np.save(save_path / f"{text_file}.npy", vectors)

    (save_path / f"{text_file}.chunks.json").write_text(
        json.dumps(chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return vectors


root = Path.home() / "Desktop/AlgoTrading"
model_path = root / "Models/BGE-M3"
save_path = root / "tests/data/embeddings"
audio_path = root / "data/audio"
transcrip_path = root / "data/transcripts"
save_path.mkdir(parents = True, exist_ok = True)
metadata = pd.read_json(audio_path / "metadataes.json")
text_ids = metadata["id"].dropna().unique()

model = download_model(model_path)
##########
tests = json.loads(
    (root / "tests/retrieval_tests.json").read_text(encoding="utf-8")
)

query_vectors = model.encode(
    [test["question"] for test in tests],
    batch_size=2,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(f"Вопросов: {len(tests)}")  # 52
#####################

query_vectors = model.encode(
    [test["question"] for test in tests],
    batch_size=2,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

configs = [
    (chunk_size, overlap)
    for chunk_size in range(192, 1024, 30)
    for overlap in [0, chunk_size // 8, chunk_size // 4]
]
results = []

for chunk_size, overlap in configs:
    variant_path = (
        save_path
        / f"chunk{chunk_size}_overlap{overlap}"
    )

    for video_id in text_ids:
        embedings(
            model=model,
            text_file=video_id,
            transcrip_path=transcrip_path,
            save_path=variant_path,
            chunk_size=chunk_size,
            overlap=overlap,
        )

    evaluation = check_model(variant_path, tests, query_vectors)

    results.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "Hit@1": evaluation["hit1"],
        "Hit@3": evaluation["hit3"],
        "missing_evidence": evaluation["missing_evidence"],
    })

    (variant_path / "evaluation.json").write_text(
        json.dumps(evaluation, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

comparison = pd.DataFrame(results).sort_values(
    ["Hit@3", "Hit@1"],
    ascending=False,
)

print(comparison.to_string(index=False))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Вопросов: 52
 chunk_size  overlap    Hit@1    Hit@3  missing_evidence
       1002        0 0.442308 0.961538                 0
        912        0 0.346154 0.903846                 0
        372       46 0.538462 0.884615                 0
        942      117 0.403846 0.884615                 0
        912      114 0.365385 0.884615                 0
        312        0 0.596154 0.865385                 0
        972      243 0.461538 0.865385                 0
        972      121 0.423077 0.865385                 0
        672      168 0.576923 0.846154                 0
        372       93 0.519231 0.846154                 0
        432      108 0.519231 0.846154                 0
        792      198 0.500000 0.846154                 0
        582       72 0.480769 0.846154                 0
        672       84 0.480769 0.846154                 0
       1002      250 0.461538 0.846154                 0
        612      153 0.442308 0.846154                 0
        552       